In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

# ============================================================
# 0. DB
# ============================================================

DB_PATH = Path("/Users/muna/Hana_research/data/db/Hana_Research.db")

conn = sqlite3.connect(DB_PATH)

print("DB:", DB_PATH)
print()


# ============================================================
# 1. 対象テーブルの構造を確認
# ============================================================

tables = [
    "tolvaptan_study",
    "Patient_Master",
    "karte",
    "lab_raw",
    "lab_clean",
    "item_master",
    "ef_long",
]

print("=" * 70)
print("1. TABLE STRUCTURE")
print("=" * 70)

for table in tables:
    print(f"\n--- {table} ---")

    exists = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type='table' AND name=?
        """,
        conn,
        params=[table]
    )

    if exists.empty:
        print("TABLE NOT FOUND")
        continue

    info = pd.read_sql_query(
        f'PRAGMA table_info("{table}")',
        conn
    )

    print(info[["cid", "name", "type"]].to_string(index=False))

    count = pd.read_sql_query(
        f'SELECT COUNT(*) AS n FROM "{table}"',
        conn
    ).iloc[0, 0]

    print("Rows:", count)


# ============================================================
# 2. tolvaptan_study の現在のデータ
# ============================================================

print("\n" + "=" * 70)
print("2. TOLVAPTAN_STUDY")
print("=" * 70)

tolvaptan = pd.read_sql_query(
    """
    SELECT *
    FROM tolvaptan_study
    ORDER BY Patient_ID
    """,
    conn
)

print("Rows:", len(tolvaptan))
print("\nColumns:")
print(list(tolvaptan.columns))

print("\nFirst 20 rows:")
print(tolvaptan.head(20).to_string(index=False))


# ============================================================
# 3. Patient_Masterとの対応を確認
# ============================================================

print("\n" + "=" * 70)
print("3. PATIENT_MASTER LINKAGE")
print("=" * 70)

pm = pd.read_sql_query(
    """
    SELECT *
    FROM Patient_Master
    WHERE Patient_ID IN (
        SELECT Patient_ID
        FROM tolvaptan_study
    )
    """,
    conn
)

print("Matched Patient_Master rows:", len(pm))
print("\nColumns:")
print(list(pm.columns))

cols_to_show = [
    c for c in [
        "Patient_ID",
        "Study_ID",
        "Birth_Date",
        "First_Visit_Date",
        "Gender",
        "Age_at_Visit"
    ]
    if c in pm.columns
]

print("\nRelevant columns:")
print(pm[cols_to_show].head(20).to_string(index=False))


# ============================================================
# 4. EF_long の実データ確認
# ============================================================

print("\n" + "=" * 70)
print("4. EF_LONG")
print("=" * 70)

if "ef_long" in tables:
    ef = pd.read_sql_query(
        """
        SELECT *
        FROM ef_long
        LIMIT 30
        """,
        conn
    )

    print(ef.to_string(index=False))

    print("\nItem distribution:")
    print(
        pd.read_sql_query(
            """
            SELECT item, COUNT(*) AS n
            FROM ef_long
            GROUP BY item
            ORDER BY n DESC
            """,
            conn
        ).to_string(index=False)
    )

    print("\nFinal HF class distribution:")
    print(
        pd.read_sql_query(
            """
            SELECT final_hf_class, COUNT(*) AS n
            FROM ef_long
            GROUP BY final_hf_class
            ORDER BY n DESC
            """,
            conn
        ).to_string(index=False)
    )


# ============================================================
# 5. lab の項目名を確認
# ============================================================

print("\n" + "=" * 70)
print("5. LAB ITEM NAMES")
print("=" * 70)

for table in ["lab_clean", "lab_raw", "item_master"]:

    if table not in tables:
        continue

    print(f"\n--- {table} ---")

    # まず列名
    info = pd.read_sql_query(
        f'PRAGMA table_info("{table}")',
        conn
    )

    print("Columns:")
    print(list(info["name"]))

    # item関連の列を探す
    item_cols = [
        c for c in info["name"]
        if any(x in c.lower() for x in
               ["item", "test", "name", "code"])
    ]

    print("Possible item columns:", item_cols)

    # item_code / item_name があれば一覧を取得
    code_col = next(
        (c for c in ["item_code", "Item_Code", "ITEM_CODE"] if c in info["name"].values),
        None
    )

    name_col = next(
        (c for c in ["item_name", "Item_Name", "ITEM_NAME"] if c in info["name"].values),
        None
    )

    if code_col or name_col:

        select_cols = []

        if code_col:
            select_cols.append(f'"{code_col}"')

        if name_col:
            select_cols.append(f'"{name_col}"')

        q = f"""
            SELECT {", ".join(select_cols)}, COUNT(*) AS n
            FROM "{table}"
            GROUP BY {", ".join(select_cols)}
            ORDER BY n DESC
            LIMIT 200
        """

        items = pd.read_sql_query(q, conn)

        print("\nItem list:")
        print(items.to_string(index=False))


# ============================================================
# 6. 重要な採血項目を名前から検索
# ============================================================

print("\n" + "=" * 70)
print("6. SEARCH LAB ITEMS")
print("=" * 70)

keywords = [
    "Na",
    "ナトリウム",
    "Sodium",
    "K",
    "カリウム",
    "BUN",
    "尿素窒素",
    "Cr",
    "クレアチニン",
    "eGFR",
    "AST",
    "ALT",
    "Alb",
    "アルブミン",
    "T-Bil",
    "ビリルビン",
    "BNP",
    "NT-proBNP",
    "Hb",
    "ヘモグロビン",
    "Hct",
    "WBC",
    "白血球",
    "Plt",
    "血小板",
]

if "item_master" in tables:

    info = pd.read_sql_query(
        'PRAGMA table_info("item_master")',
        conn
    )
    code_col = next(
        (c for c in ["item_code", "Item_Code", "ITEM_CODE"] if c in info["name"].values),
        None
    )

    name_col = next(
        (c for c in ["item_name", "Item_Name", "ITEM_NAME"] if c in info["name"].values),
        None
    )

    # item_masterの全データを少量確認
    item_master = pd.read_sql_query(
        'SELECT * FROM item_master LIMIT 20',
        conn
    )

    print("item_master columns:")
    print(list(info["name"]))

    code_col = next(
        (c for c in ["item_code", "Item_Code", "ITEM_CODE"] if c in info["name"].values),
        None
    )

    name_col = next(
        (c for c in ["item_name", "Item_Name", "ITEM_NAME"] if c in info["name"].values),
        None
    )

    # item_masterの全データを少量確認
    item_master = pd.read_sql_query(
        'SELECT * FROM item_master LIMIT 20',
        conn
    )

    print("\nFirst rows of item_master:")
    print(item_master.to_string(index=False))

    # item_masterの全データを少量確認
    item_master = pd.read_sql_query(
        'SELECT * FROM item_master LIMIT 20',
        conn
    )

    print("\nFirst rows of item_master:")
    print(item_master.to_string(index=False))


# --- keywordsを使った実際の検索 ---
    if name_col or code_col:

        search_cols = [c for c in [name_col, code_col] if c is not None]

        # 各カラム×各キーワードでLIKE条件を組み立てる
        where_clause = " OR ".join(
            f'"{col}" LIKE ?' for col in search_cols for _ in keywords
        )
        params = [f"%{kw}%" for _ in search_cols for kw in keywords]

        select_cols_str = ", ".join(f'"{c}"' for c in search_cols)

        q = f"""
            SELECT {select_cols_str}
            FROM item_master
            WHERE {where_clause}
        """

        matched = pd.read_sql_query(q, conn, params=params)

        print(f"\nMatched items ({len(matched)} rows):")
        print(matched.drop_duplicates().to_string(index=False))

    else:
        print("\n注意: item_name / item_code に該当する列が見つかりませんでした。"
              "実際の列名をPRAGMA結果から確認して、code_col / name_colの候補リストを調整してください。")

変更点の説明

search_cols（name_colとcode_colのうち存在するもの）それぞれに対して、keywordsの各語をLIKE '%語%'で検索するWHERE句を動的に組み立てました
パラメータ化クエリ（?プレースホルダ + params）にしているので、SQLインジェクションの心配もなくキーワード内の特殊文字も安全に扱えます
一致した行を重複除去（drop_duplicates）して表示します
name_colもcode_colも見つからなかった場合は、無言でスキップせず注意メッセージを出すようにしました（元のコードだと該当カラムがない場合に何も起きず気づきにくいため）

もしitem_masterの実際のカラム名（PRAGMA結果）が分かれば、code_col/name_colの決め打ちリストも実データに合わせて調整できます。

# ============================================================
# 8. 終了
# ============================================================

conn.close()

print("\n" + "=" * 70)
print("STEP 1 FINISHED")
print("No database tables were modified.")
print("=" * 70)